# 3. Check the segments and write the manifests

1. **Downloads.** Every segment is downloaded the way users get it (`../scripts/download_*.py`)
   and its first and last frames are compared with the broadcast's (`work/ends/`). The archives'
   streams do not always seek to the frame the broadcast had there: such a segment's bounds are
   shifted by the frames it was off. A broadcast the archive no longer serves at the same height,
   or serves as another edit, is excluded, with the reason.
2. **Checks**, whose outcome a human writes to `decisions/`:
   - PrimaTV: videos where the interpreter was seen again outside the block (`look_again`).
   - Digi24: the mean frame of every segment, sorted by how far it is from the average one, to
     drop segments that are not interpretation; and an identity check every 2 s, to cut the
     start or end of segments where the box shows something else.
3. **Manifests**: the segments, shifted as measured, minus the decisions, in
   `work/release/manifests/` (4_labels adds the splits), and the excluded broadcasts in
   `work/release/docs/excluded_broadcasts.json`.

`REPLAY = True` skips 1 and 2 and builds the manifests from the original run (`frozen/`).

In [ ]:
import json
import sys
from datetime import date, timedelta
from functools import partial
from pathlib import Path
from types import SimpleNamespace

import cv2
import numpy as np

from common import CHANNELS, CROP, DECISIONS, FPS, FROZEN, REPO, WORK, Video, digi24_era, read, write

REPLAY = True
SRC = FROZEN if REPLAY else WORK
segments = read(SRC / 'segments.json')
broadcasts = read(SRC / 'broadcasts.json')
MP4S = WORK / 'mp4s'                       # the segments as users download them
sys.path.insert(0, str(REPO / 'scripts'))


def similarity(a, b):
    """Normalized cross-correlation of two images' gray levels (1: same picture)."""
    a, b = (np.asarray(x, np.float64).mean(-1).ravel() for x in (a, b))
    a, b = a - a.mean(), b - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

## 1. Downloads

`sources(channel, key, height, crop)` gives the archive's videos that may hold the broadcast
(for a vid2 file, the protv.ro episodes of the days around it), each with `grab(start, end,
dest)`, which cuts frames [start, end) of it into `dest` exactly as the user's downloader does. Around
each end of a segment, the downloaded frames are compared with the broadcast's at shifts of up to
3 frames (gray-level correlation; the download is re-encoded, so even the right frame only
reaches about 0.98, and a static picture's neighbours as much: the start, usually a transition,
decides).

In [ ]:
import requests


def sources(channel, key, height, crop):
    """(page, grab) for each archive video that may hold the broadcast, at the broadcast's height."""
    if channel == 'ProTV':
        import download_protv as d
        day, name = key.split('/')
        listing = broadcasts['ProTV']['protv.ro']
        if name.endswith('-2'):       # a vid2 file, stored on or about the day it aired
            near = [(date.fromisoformat(day) + timedelta(n)).isoformat() for n in (0, -1, 1, -2, 2)]
            pages = [u for x in near for u in listing.get(x, [])]
        else:
            pages = [u for u in listing.get(day, []) if u.rstrip('/').rsplit('/', 1)[1].split('-')[0] == name]
        for url in pages:
            try:
                source = d.resolve_manifest_url(requests.Session(), url)
                stream = d.pick_video_stream(source, height)
            except RuntimeError:
                continue
            t0 = d.stream_start_time(source, stream)
            yield url, partial(lambda src, st, t, s, e, dest: d.grab_segment(src, st, t, {'start': s, 'end': e}, crop, dest),
                               source, stream, t0)
    elif channel == 'PrimaTV':
        import download_prima as d
        args = SimpleNamespace(yt_dlp=str(Path(sys.executable).with_name('yt-dlp')), cookies='cookies.txt',
                               js_runtime='node', remote_components=True)
        item = {'id': key, 'url': f'https://www.youtube.com/watch?v={key}', 'source_height': height}
        path = d.download_stream(args, item, WORK / 'downloads')
        t0 = d.stream_start_time(path)
        yield item['url'], lambda s, e, dest: d.grab_segment(path, t0, {'start': s, 'end': e}, crop, dest)
    else:
        import download_digi24 as d
        pages = broadcasts['Digi24']['listing'][key]
        url = read(DECISIONS / 'digi24.json')['pages'].get(key, pages[0])
        src = d.resolve_video_src(requests.Session(), url, height)
        d.check_source(src, height)
        t0 = d.stream_start_time(src)
        yield url, lambda s, e, dest: d.grab_segment(src, t0, {'start': s, 'end': e}, crop, dest)


def frames_of(path):
    video = Video(path)
    return [video[i] for i in range(video.frames)]


def shift_of(grab, ends, i, start, end, tmp, search=3):
    """The shift (in frames, -search..search) that best lines the archive's frames up with the
    broadcast's first and last 3 frames of segment i, preferring the smallest of equally good ones;
    None if no shift makes them look alike (mean correlation under 0.9)."""
    scores = {}
    for first in (start, end - 3):
        lo = max(0, first - search)
        grab(lo, first + 3 + search, tmp)
        got = frames_of(tmp)
        for d in range(-search, search + 1):
            for j in range(first, first + 3):
                if 0 <= j + d - lo < len(got):
                    scores.setdefault(d, []).append(similarity(ends[f'{i}/{j}'], got[j + d - lo]))
    mean = {d: float(np.mean(v)) for d, v in scores.items() if len(v) == 6}
    best = max(mean.values())
    if best < 0.9:
        return None
    return min((d for d, m in mean.items() if m >= best - 0.002), key=abs)


if not REPLAY:
    alignment = read(WORK / 'alignment.json') if (WORK / 'alignment.json').exists() else {c: {} for c in CHANNELS}
    tmp = WORK / 'piece.mp4'
    for channel in CHANNELS:
        for key, seg in segments[channel].items():
            if seg is None or key in alignment[channel]:
                continue
            crop = CROP[channel][digi24_era(key) if channel == 'Digi24' else seg['source_height']]
            ends = np.load(WORK / 'ends' / channel / f"{key.replace('/', '_')}.npz")
            try:
                for url, grab in sources(channel, key, seg['source_height'], crop):
                    shifts = [shift_of(grab, ends, i, s, e, tmp) for i, (s, e) in enumerate(seg['segments'])]
                    if None not in shifts:
                        break
                else:
                    raise RuntimeError("no video of the archive matches the broadcast")
                folder = key if channel == 'PrimaTV' else key.split('/')[0].replace('-', '/')
                for i, ((s, e), d) in enumerate(zip(seg['segments'], shifts)):
                    dest = MP4S / channel / folder / f'segment_{i}.mp4'
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    grab(s + d, e + d, dest)
                    assert Video(dest).frames == e - s, f'{dest}: frame count'
                alignment[channel][key] = {'url': url, 'shift': shifts}
            except Exception as error:
                alignment[channel][key] = {'unavailable': str(error)[:300]}
            write(WORK / 'alignment.json', alignment)
            for p in (WORK / 'downloads').glob(f'{key}.*'):
                p.unlink()
alignment = read(SRC / 'alignment.json')

## 2. Checks

### PrimaTV: interpreter seen again

Decide for each video listed whether to keep it; dropped ones go to `decisions/primatv.json`.

In [ ]:
for key, seg in segments['PrimaTV'].items():
    if seg and seg.get('look_again'):
        print(key, seg['look_again'])

### Digi24: mean frames

The mean of all frames of each segment. Those far from the average are shown first:
`d` next, `a` previous, `x` drop / keep again, `q` done. Dropped segments are written to
`decisions/digi24.json` (`dropped_segments`).

In [ ]:
if not REPLAY:
    means = {}
    for path in sorted(MP4S.glob('Digi24/*/*/*/segment_*.mp4')):
        total, n = None, 0
        for batch in Video(path).read():
            for frame in batch:
                total = frame.astype(np.float64) if total is None else total + frame
                n += 1
        means[path] = (total / n).clip(0, 255).astype(np.uint8)

    def padded(img, w=248, h=224):
        top, left = (h - img.shape[0]) // 2, (w - img.shape[1]) // 2
        return np.pad(img, ((top, h - img.shape[0] - top), (left, w - img.shape[1] - left), (0, 0)))
    average = np.mean([padded(m) for m in means.values()], axis=0)
    order = sorted(means, key=lambda p: -np.mean((padded(means[p]) - average) ** 2))

    decisions = read(DECISIONS / 'digi24.json')
    dropped = set(decisions['dropped_segments'])
    name = lambda p: f"{'-'.join(p.parts[-4:-1])}/{p.stem.split('_')[1]}"
    i = 0
    cv2.namedWindow('mean frame', cv2.WINDOW_NORMAL)
    while order:
        p = order[i]
        print(name(p), 'DROPPED' if name(p) in dropped else '')
        cv2.imshow('mean frame', means[p])
        key = cv2.waitKey(0)
        if key == ord('q'):
            break
        if key == ord('x'):
            dropped ^= {name(p)}
        i = (i + (-1 if key == ord('a') else 1)) % len(order)
    cv2.destroyAllWindows()
    decisions['dropped_segments'] = sorted(dropped)
    write(DECISIONS / 'digi24.json', decisions)

### Digi24: who is in the box

Every 2 s of every segment, the face in the box is matched against reference frames of each
interpreter: 20 evenly spaced frames of each of the hand-checked segments in
`decisions/digi24.json` (`reference_segments`). FaceNet (MTCNN, InceptionResnetV1 trained on
VGGFace2) gives the distance to an interpreter as the mean of the 5 nearest reference frames.
A check is a hit if the nearest interpreter is the day's interpreter (the most frequent nearest
one), at distance 1.0 or less and 0.05 closer than the next. A check that is not a hit is done
again with dlib (face_recognition: distance 0.55 or less, 0.03 closer than the next). A segment
is flagged where a check and the two 1 s after it all miss; on 32 hand-checked segments this
missed 0.3% of the right interpreter's frames and accepted no other interpreter.

The flagged segments are listed with an image of the 3 frames (`work/flags/`); a human decides
where to cut (`cuts`: frames to cut from the start and from the end).

In [ ]:
KNN = 5

if not REPLAY:
    import face_recognition
    import torch
    from collections import Counter
    from facenet_pytorch import MTCNN, InceptionResnetV1

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    facenet = InceptionResnetV1(pretrained='vggface2').eval().to(device)
    mtcnn = MTCNN(image_size=160, margin=14, min_face_size=20, device=device, select_largest=True)

    def facenet_embeddings(images):
        """Per image: its face's embedding, or None if no face is found."""
        out = []
        for j in range(0, len(images), 128):
            chunk = images[j:j + 128]
            boxes, _ = mtcnn.detect(chunk)
            faces = [None if b is None else mtcnn.extract(im, b[:1], None) for im, b in zip(chunk, boxes)]
            faces = [None if f is None else (f[0] if f.dim() == 4 else f) for f in faces]
            found = [f for f in faces if f is not None]
            with torch.inference_mode():
                emb = iter(facenet(torch.stack(found).to(device)).cpu() if found else [])
            out += [None if f is None else next(emb) for f in faces]
        return out

    def dlib_embedding(image):
        image = np.ascontiguousarray(image)
        boxes = face_recognition.face_locations(image, 1)
        if not boxes:
            return None
        biggest = max(boxes, key=lambda b: (b[2] - b[0]) * (b[1] - b[3]))
        return face_recognition.face_encodings(image, [biggest])[0]

    def every_second(path):
        video = Video(path)
        return {t: video[t * FPS] for t in range(0, (video.frames + FPS - 1) // FPS)}

    reference = {'facenet': {}, 'dlib': {}}
    for seg, interpreter in read(DECISIONS / 'digi24.json')['reference_segments'].items():
        day, i = seg.split('/')
        video = Video(MP4S / 'Digi24' / day.replace('-', '/') / f'segment_{i}.mp4')
        frames = [video[int((k + 0.5) * video.frames / 20)] for k in range(20)]
        reference['facenet'].setdefault(interpreter, []).extend(e for e in facenet_embeddings(frames) if e is not None)
        reference['dlib'].setdefault(interpreter, []).extend(e for e in map(dlib_embedding, frames) if e is not None)
    reference = {m: {n: np.stack([np.asarray(e) for e in es]) for n, es in r.items()} for m, r in reference.items()}

    def distances(model, embedding):
        if embedding is None:
            return None
        return {n: float(np.sort(np.linalg.norm(r - np.asarray(embedding), axis=1))[:KNN].mean())
                for n, r in reference[model].items()}

    def hit(d, expected, threshold, margin):
        if d is None or min(d, key=d.get) != expected or d[expected] > threshold:
            return False
        return min(v for n, v in d.items() if n != expected) - d[expected] >= margin

    identity, flags = {}, {}
    (WORK / 'flags').mkdir(parents=True, exist_ok=True)
    for day, seg in segments['Digi24'].items():
        if seg is None or 'unavailable' in alignment['Digi24'].get(day, {}):
            continue
        checks = {}
        for i in range(len(seg['segments'])):
            frames = every_second(MP4S / 'Digi24' / day.replace('-', '/') / f'segment_{i}.mp4')
            checks[i] = (frames, dict(zip(frames, map(lambda e: distances('facenet', e), facenet_embeddings(list(frames.values()))))))
        nearest = Counter(min(d, key=d.get) for _, c in checks.values() for d in c.values() if d)
        identity[day] = expected = nearest.most_common(1)[0][0] if nearest else None
        for i, (frames, facenet_d) in checks.items():
            ok = {t: hit(d, expected, 1.0, 0.05) or hit(distances('dlib', dlib_embedding(frames[t])), expected, 0.55, 0.03)
                  for t, d in facenet_d.items()}
            for t in sorted(ok):
                if t % 2 == 0 and all(x in ok and not ok[x] for x in (t, t + 1, t + 2)):
                    flags[f'{day}/{i}'] = t * FPS
                    cv2.imwrite(str(WORK / 'flags' / f'{day}_{i}_t{t}.png'),
                                np.hstack([frames[x] for x in (t, t + 1, t + 2)])[:, :, ::-1])
                    break
    write(WORK / 'identity.json', {'interpreter': identity, 'flags': flags})
    print(len(flags), 'segments flagged:', flags)

## 3. Manifests

One item per broadcast that the archive still serves, with the segments as measured in step 1,
minus the dropped ones; the interpreter box (`crop`) is the one of the broadcast's height.
Item order: PrimaTV and ProTV list the broadcasts found only by a second search of the archive
first (as the original run's first manifests did; the split draw in 4_labels depends on the
order), then by video id or date.

In [ ]:
digi24 = read(DECISIONS / 'digi24.json')
primatv = read(DECISIONS / 'primatv.json')
identity = {} if REPLAY else read(WORK / 'identity.json')['interpreter']
items, excluded = {c: [] for c in CHANNELS}, {'PrimaTV': [], 'ProTV': []}
later = lambda channel: (lambda k: (not alignment[channel].get(k, {}).get('found_later'), k))


def as_segments(ranges, shift=None):
    shift = shift or [0] * len(ranges)
    return [{'index': i, 'start': s + d, 'end': e + d} for i, ((s, e), d) in enumerate(zip(ranges, shift))]


for key in sorted(segments['PrimaTV'], key=later('PrimaTV')):
    seg, where = segments['PrimaTV'][key], alignment['PrimaTV'].get(key, {})
    if seg is None or key in primatv['dropped']:
        continue
    url = f'https://www.youtube.com/watch?v={key}'
    if 'unavailable' in where:
        excluded['PrimaTV'].append({'id': key, 'url': url, 'interpreter': seg['interpreter'], 'source_height': None,
                                    'crop': None, 'segments': [], 'unrecoverable': True, 'reason': where['unavailable']})
        continue
    items['PrimaTV'].append({'id': key, 'url': url, 'interpreter': seg['interpreter'],
                             'source_height': seg['source_height'], 'crop': CROP['PrimaTV'][seg['source_height']],
                             'segments': as_segments(seg['segments'], where['shift'])})

same_day = {}
for key in sorted(segments['ProTV']):
    same_day.setdefault(key.split('/')[0], []).append(key)
for key in sorted(segments['ProTV'], key=later('ProTV')):
    seg, where = segments['ProTV'][key], alignment['ProTV'][key]
    if seg is None:
        continue
    day = key.split('/')[0]
    if 'unavailable' in where:
        n = same_day[day]
        excluded['ProTV'].append({'date': day, 'name': 'video.mp4' if len(n) == 1 else f'video_{n.index(key)}.mp4',
                                  'url': where.get('url'), 'interpreter': seg['interpreter'], 'crop': None,
                                  'segments': as_segments(seg['segments']), 'unrecoverable': True,
                                  'reason': where['unavailable']})
        continue
    items['ProTV'].append({'date': day, 'url': where['url'], 'interpreter': seg['interpreter'],
                           'crop': CROP['ProTV'][seg['source_height']], 'source_height': seg['source_height'],
                           'segments': as_segments(seg['segments'], where['shift'])})
assert len({it['date'] for it in items['ProTV']}) == len(items['ProTV']), 'one ProTV broadcast per day'

for day in sorted(segments['Digi24']):
    seg = segments['Digi24'][day]
    if seg is None or day in digi24['dropped_days']:
        continue
    pages = broadcasts['Digi24']['listing'][day]
    url = digi24['pages'].get(day, pages[0])
    kept = []
    for s in as_segments(seg['segments'], alignment.get('Digi24', {}).get(day, {}).get('shift')):
        if f"{day}/{s['index']}" in digi24['dropped_segments']:
            continue
        first, last = digi24['cuts'].get(f"{day}/{s['index']}", (0, 0))
        kept.append({**s, 'start': s['start'] + first, 'end': s['end'] - last})
    items['Digi24'].append({'date': day, 'url': url, 'video_src': broadcasts['Digi24']['video_src'][url],
                            'interpreter': seg['interpreter'] or identity.get(day), 'crop': CROP['Digi24'][digi24_era(day)],
                            'source_height': seg['source_height'], 'segments': kept})

write(WORK / 'release' / 'manifests.json', items)
write(WORK / 'release' / 'docs' / 'excluded_broadcasts.json', {
    'description': 'Broadcasts that were in the source lists but are not in RSL-News, because they could not be '
                   'recovered from the public archives; each keeps its manifest entry and the reason. Taken from the '
                   'download manifests of Sep 28-29, 2026. Segment ranges are [start, end) with end exclusive, as in '
                   'manifests/ (the original lists stored PrimaTV and ProTV ends inclusive).',
    **excluded}, indent=2, newline=False)
print({c: (len(items[c]), sum(len(it['segments']) for it in items[c])) for c in CHANNELS},
      {c: len(v) for c, v in excluded.items()})